# Notebook 3 — Jointures, agrégations, Parquet et PostgreSQL

In [1]:
from collections import Counter
import os

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("TradeCorp - Transformations")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

TMP_PATH = "/home/jovyan/data/tmp"
OUTPUT_PATH = "/home/jovyan/data/output"

def lire_parquet(nom):
    return spark.read.parquet(
        f"{TMP_PATH}/{nom}.parquet"
    )

df_customers = lire_parquet("customers")
df_orders = lire_parquet("orders")
df_order_details = lire_parquet("order_details")
df_products = lire_parquet("products")
df_categories = lire_parquet("categories")
df_suppliers = lire_parquet("suppliers")
df_employees = lire_parquet("employees")
df_shippers = lire_parquet("shippers")

print("DataFrames Parquet chargés avec succès.")

DataFrames Parquet chargés avec succès.


In [2]:
dataframes = {
    "customers": df_customers,
    "orders": df_orders,
    "order_details": df_order_details,
    "products": df_products,
    "categories": df_categories,
    "suppliers": df_suppliers,
    "employees": df_employees,
    "shippers": df_shippers,
}

for nom, df in dataframes.items():
    print(nom, "=>", df.count(), "ligne(s)")

customers => 91 ligne(s)
orders => 809 ligne(s)
order_details => 2155 ligne(s)
products => 77 ligne(s)
categories => 8 ligne(s)
suppliers => 29 ligne(s)
employees => 9 ligne(s)
shippers => 6 ligne(s)


## Q21 Jointures orders et customers

In [3]:
df_orders_customers = df_orders.alias("o").join(df_customers.alias("c"),F.col("o.customer_id")==F.col("c.customer_id"),"inner",).select("order_id","company_name","country","order_date","freight")

In [4]:
df_orders_customers.show(10, truncate=False)

+--------+-------------------------+-----------+----------+-------+
|order_id|company_name             |country    |order_date|freight|
+--------+-------------------------+-----------+----------+-------+
|10248   |Vins et alcools Chevalier|FRANCE     |1996-07-04|32.38  |
|10249   |Toms Spezialitäten       |GERMANY    |1996-07-05|11.61  |
|10250   |Hanari Carnes            |BRAZIL     |1996-07-08|65.83  |
|10251   |Victuailles en stock     |FRANCE     |1996-07-08|41.34  |
|10252   |Suprêmes délices         |BELGIUM    |1996-07-09|51.3   |
|10253   |Hanari Carnes            |BRAZIL     |1996-07-10|58.17  |
|10254   |Chop-suey Chinese        |SWITZERLAND|1996-07-11|22.98  |
|10255   |Richter Supermarkt       |SWITZERLAND|1996-07-12|148.33 |
|10256   |Wellington Importadora   |BRAZIL     |1996-07-15|13.97  |
|10257   |HILARION-Abastos         |VENEZUELA  |1996-07-16|81.91  |
+--------+-------------------------+-----------+----------+-------+
only showing top 10 rows


# Q22 Jointure order details et products 

In [5]:
df_order_details_products = df_order_details.alias("od").join(df_products.alias("p"), F.col("od.product_id")==F.col("p.product_id"),"inner",).select("product_name","category_id","unit_price")

In [6]:
df_order_details_products.show(10, truncate=False)

+--------------------------------+-----------+----------+
|product_name                    |category_id|unit_price|
+--------------------------------+-----------+----------+
|Queso Cabrales                  |4          |21.0      |
|Singaporean Hokkien Fried Mee   |5          |14.0      |
|Mozzarella di Giovanni          |4          |34.8      |
|Tofu                            |7          |23.25     |
|Manjimup Dried Apples           |7          |53.0      |
|Jack's New England Clam Chowder |8          |9.65      |
|Manjimup Dried Apples           |7          |53.0      |
|Louisiana Fiery Hot Pepper Sauce|2          |21.05     |
|Gustaf's Knäckebröd             |5          |21.0      |
|Ravioli Angelo                  |5          |19.5      |
+--------------------------------+-----------+----------+
only showing top 10 rows


## Q23 Jointure products et categories 

In [7]:
df_products_categories = (
    df_products.alias("p")
    .join(
        df_categories.alias("c"),
        F.col("p.category_id") == F.col("c.category_id"),
        "inner",
    )
    .select(
        F.col("p.*"),
        F.col("c.category_name"),
        F.col("c.description").alias(
            "category_description"
        ),
    )
)

df_products_categories.select(
    "product_id",
    "product_name",
    "category_name",
    "category_description",
).show(10, truncate=False)

+----------+-------------------------------+-------------+----------------------------------------------------------+
|product_id|product_name                   |category_name|category_description                                      |
+----------+-------------------------------+-------------+----------------------------------------------------------+
|1         |Chai                           |Beverages    |Soft drinks, coffees, teas, beers, and ales               |
|2         |Chang                          |Beverages    |Soft drinks, coffees, teas, beers, and ales               |
|3         |Aniseed Syrup                  |Condiments   |Sweet and savory sauces, relishes, spreads, and seasonings|
|4         |Chef Anton's Cajun Seasoning   |Condiments   |Sweet and savory sauces, relishes, spreads, and seasonings|
|5         |Chef Anton's Gumbo Mix         |Condiments   |Sweet and savory sauces, relishes, spreads, and seasonings|
|6         |Grandma's Boysenberry Spread   |Condiments  

## Q24A — Jointure complète sans renommage préalable

In [8]:
df_products_categories_brut = (
    df_products
    .join(df_categories, "category_id", "inner")
)

df_jointure_brute = (
    df_order_details
    .join(df_orders, "order_id", "inner")
    .join(df_customers, "customer_id", "inner")
    .join(
        df_products_categories_brut,
        "product_id",
        "inner",
    )
    .join(df_employees, "employee_id", "left")
    .join(df_shippers, "shipper_id", "left")
)

compteur_colonnes = Counter(
    df_jointure_brute.columns
)

colonnes_dupliquees = {
    colonne: nombre
    for colonne, nombre in compteur_colonnes.items()
    if nombre > 1
}

print("Colonnes présentes plusieurs fois :")
print(colonnes_dupliquees)

Colonnes présentes plusieurs fois :
{'company_name': 2, 'city': 2, 'country': 2, 'phone': 2}


## Q24B — Renommage des colonnes et reconstruction

In [9]:
def renommer_colonnes(df, correspondances):
    resultat = df

    for ancien_nom, nouveau_nom in correspondances.items():
        if ancien_nom in resultat.columns:
            resultat = resultat.withColumnRenamed(
                ancien_nom,
                nouveau_nom
            )

    return resultat

In [10]:
df_customers_renamed = renommer_colonnes(
    df_customers,
    {
        "company_name": "customer_company_name",
        "contact_name": "customer_contact_name",
        "contact_title": "customer_contact_title",
        "address": "customer_address",
        "city": "customer_city",
        "region": "customer_region",
        "postal_code": "customer_postal_code",
        "country": "customer_country",
        "phone": "customer_phone",
        "fax": "customer_fax",
    },
)

In [11]:
df_products_renamed = renommer_colonnes(
    df_products,
    {
        "unit_price": "product_unit_price",
    },
)

df_categories_renamed = renommer_colonnes(
    df_categories,
    {
        "description": "category_description",
    },
)

df_products_enriched = (
    df_products_renamed
    .join(
        df_categories_renamed,
        "category_id",
        "inner",
    )
)

In [12]:
df_employees_renamed = renommer_colonnes(
    df_employees,
    {
        "first_name": "employee_first_name",
        "last_name": "employee_last_name",
        "title": "employee_title",
        "hire_date": "employee_hire_date",
        "city": "employee_city",
        "country": "employee_country",
        "full_name": "employee_full_name",
    },
)

In [13]:
df_shippers_renamed = renommer_colonnes(
    df_shippers,
    {
        "company_name": "shipper_name",
        "phone": "shipper_phone",
    },
)

In [14]:
df_orders_enriched = (
    df_order_details
    .join(df_orders, "order_id", "inner")
    .join(
        df_customers_renamed,
        "customer_id",
        "inner",
    )
    .join(
        df_products_enriched,
        "product_id",
        "inner",
    )
    .join(
        df_employees_renamed,
        "employee_id",
        "left",
    )
    .join(
        df_shippers_renamed,
        "shipper_id",
        "left",
    )
)

df_orders_enriched.cache()

print(
    "Nombre de lignes enrichies :",
    df_orders_enriched.count()
)

print(
    "Nombre de colonnes :",
    len(df_orders_enriched.columns)
)

Nombre de lignes enrichies : 2082
Nombre de colonnes : 52


In [15]:
compteur_final = Counter(
    df_orders_enriched.columns
)

doublons_finaux = {
    colonne: nombre
    for colonne, nombre in compteur_final.items()
    if nombre > 1
}

print("Colonnes dupliquées restantes :", doublons_finaux)

Colonnes dupliquées restantes : {}


## Q25 — Chiffre d'affaires par client

In [16]:
df_ca_client = (
    df_orders_enriched
    .groupBy("customer_company_name")
    .agg(
        F.round(
            F.sum("sous_total"),
            2
        ).alias("ca_total")
    )
    .orderBy(F.desc("ca_total"))
)

df_ca_client.show(10, truncate=False)

+----------------------------+---------+
|customer_company_name       |ca_total |
+----------------------------+---------+
|QUICK-Stop                  |110277.32|
|Save-a-lot Markets          |104361.96|
|Ernst Handel                |94976.09 |
|Hungry Owl All-Night Grocers|49979.91 |
|Rattlesnake Canyon Grocery  |49842.08 |
|Hanari Carnes               |32841.37 |
|Königlich Essen             |30908.39 |
|Folk och fä HB              |29567.57 |
|Mère Paillarde              |28872.2  |
|White Clover Markets        |27363.61 |
+----------------------------+---------+
only showing top 10 rows


In [17]:
#df_orders_enriched.show(10, truncate=False)

# Q26 CA par categorie 

In [18]:
df_ca_categorie = (
    df_orders_enriched
    .groupBy("category_name")
    .agg(
        F.round(
            F.sum("sous_total"),
            2
        ).alias("ca_total"),
        F.countDistinct("product_id").alias(
            "nombre_produits_vendus"
        ),
    )
    .orderBy(F.desc("ca_total"))
)

df_ca_categorie.show(truncate=False)

+--------------+---------+----------------------+
|category_name |ca_total |nombre_produits_vendus|
+--------------+---------+----------------------+
|Beverages     |262572.5 |12                    |
|Dairy Products|230951.2 |10                    |
|Confections   |164672.1 |13                    |
|Meat/Poultry  |162132.23|6                     |
|Seafood       |130070.16|12                    |
|Condiments    |105047.25|12                    |
|Produce       |93630.8  |5                     |
|Grains/Cereals|90779.59 |7                     |
+--------------+---------+----------------------+



# Q27 CA par mois 

In [19]:
df_ca_mensuel = (
    df_orders_enriched
    .withColumn(
        "mois",
        F.date_trunc("month", F.col("order_date"))
    )
    .groupBy("mois")
    .agg(
        F.round(
            F.sum("sous_total"),
            2
        ).alias("ca_mensuel")
    )
    .orderBy("mois")
)

df_ca_mensuel.show(50, truncate=False)

+-------------------+----------+
|mois               |ca_mensuel|
+-------------------+----------+
|1996-07-01 00:00:00|27861.9   |
|1996-08-01 00:00:00|25485.28  |
|1996-09-01 00:00:00|26381.4   |
|1996-10-01 00:00:00|37515.73  |
|1996-11-01 00:00:00|45600.05  |
|1996-12-01 00:00:00|45239.63  |
|1997-01-01 00:00:00|61258.08  |
|1997-02-01 00:00:00|38483.64  |
|1997-03-01 00:00:00|38547.23  |
|1997-04-01 00:00:00|53032.95  |
|1997-05-01 00:00:00|53781.3   |
|1997-06-01 00:00:00|36362.82  |
|1997-07-01 00:00:00|51020.88  |
|1997-08-01 00:00:00|47287.68  |
|1997-09-01 00:00:00|55629.27  |
|1997-10-01 00:00:00|66749.24  |
|1997-11-01 00:00:00|43533.81  |
|1997-12-01 00:00:00|71398.44  |
|1998-01-01 00:00:00|94222.13  |
|1998-02-01 00:00:00|99415.29  |
|1998-03-01 00:00:00|104854.19 |
|1998-04-01 00:00:00|110488.89 |
|1998-05-01 00:00:00|5706.0    |
+-------------------+----------+



## Q28 — Performance par employé

In [20]:
df_ca_employe = (
    df_orders_enriched
    .groupBy("employee_full_name")
    .agg(
        F.round(
            F.sum("sous_total"),
            2
        ).alias("ca_total")
    )
)

In [21]:
df_commandes_employe = (
    df_orders_enriched
    .select(
        "employee_full_name",
        "order_id",
        "order_date",
        "shipped_date",
    )
    .dropDuplicates(
        ["employee_full_name", "order_id"]
    )
    .withColumn(
        "delai_livraison_jours",
        F.datediff(
            F.col("shipped_date"),
            F.col("order_date")
        )
    )
    .groupBy("employee_full_name")
    .agg(
        F.countDistinct("order_id").alias(
            "nombre_commandes"
        ),
        F.round(
            F.avg("delai_livraison_jours"),
            2
        ).alias("delai_moyen_jours"),
    )
)

In [22]:
df_performance_employes = (
    df_commandes_employe
    .join(
        df_ca_employe,
        "employee_full_name",
        "inner",
    )
    .orderBy(F.desc("ca_total"))
)

df_performance_employes.show(
    truncate=False
)

+------------------+----------------+-----------------+---------+
|employee_full_name|nombre_commandes|delai_moyen_jours|ca_total |
+------------------+----------------+-----------------+---------+
|Margaret Peacock  |151             |8.82             |225763.74|
|Janet Leverling   |127             |8.43             |202812.88|
|Nancy Davolio     |120             |7.76             |187277.43|
|Andrew Fuller     |93              |8.05             |162769.78|
|Laura Callahan    |100             |8.68             |123842.7 |
|Robert King       |69              |8.38             |119619.25|
|Anne Dodsworth    |42              |10.86            |76450.09 |
|Michael Suyama    |65              |9.09             |72527.65 |
|Steven Buchanan   |42              |7.02             |68792.31 |
+------------------+----------------+-----------------+---------+



## Q29 — Window function : classement des produits

In [23]:
df_ca_produit = (
    df_orders_enriched
    .groupBy(
        "category_name",
        "product_id",
        "product_name",
    )
    .agg(
        F.round(
            F.sum("sous_total"),
            2
        ).alias("ca_produit")
    )
)

fenetre_rang = (
    Window
    .partitionBy("category_name")
    .orderBy(F.desc("ca_produit"))
)

df_classement_produits = (
    df_ca_produit
    .withColumn(
        "rang_categorie",
        F.dense_rank().over(fenetre_rang)
    )
    .orderBy(
        "category_name",
        "rang_categorie"
    )
)

df_classement_produits.show(
    100,
    truncate=False
)

+--------------+----------+--------------------------------+----------+--------------+
|category_name |product_id|product_name                    |ca_produit|rang_categorie|
+--------------+----------+--------------------------------+----------+--------------+
|Beverages     |38        |Côte de Blaye                   |141396.74 |1             |
|Beverages     |43        |Ipoh Coffee                     |22119.1   |2             |
|Beverages     |76        |Lakkalikööri                    |15729.84  |3             |
|Beverages     |2         |Chang                           |15354.66  |4             |
|Beverages     |35        |Steeleye Stout                  |13212.0   |5             |
|Beverages     |39        |Chartreuse verte                |12260.34  |6             |
|Beverages     |1         |Chai                            |12176.1   |7             |
|Beverages     |70        |Outback Lager                   |10528.65  |8             |
|Beverages     |75        |Rhönbräu Kloster

In [24]:
df_classement_produits.filter(
    F.col("rang_categorie") <= 5
).show(100, truncate=False)

+--------------+----------+--------------------------------+----------+--------------+
|category_name |product_id|product_name                    |ca_produit|rang_categorie|
+--------------+----------+--------------------------------+----------+--------------+
|Beverages     |38        |Côte de Blaye                   |141396.74 |1             |
|Beverages     |43        |Ipoh Coffee                     |22119.1   |2             |
|Beverages     |76        |Lakkalikööri                    |15729.84  |3             |
|Beverages     |2         |Chang                           |15354.66  |4             |
|Beverages     |35        |Steeleye Stout                  |13212.0   |5             |
|Condiments    |63        |Vegie-spread                    |16701.1   |1             |
|Condiments    |61        |Sirop d'érable                  |14238.6   |2             |
|Condiments    |65        |Louisiana Fiery Hot Pepper Sauce|13869.91  |3             |
|Condiments    |8         |Northwoods Cranb

## Q30 — Chiffre d’affaires cumulé

In [25]:
fenetre_cumul = (
    Window
    .orderBy("mois")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow,
    )
)

df_ca_cumule = (
    df_ca_mensuel
    .withColumn(
        "ca_cumule",
        F.round(
            F.sum("ca_mensuel").over(fenetre_cumul),
            2
        )
    )
)

df_ca_cumule.show(50, truncate=False)

+-------------------+----------+----------+
|mois               |ca_mensuel|ca_cumule |
+-------------------+----------+----------+
|1996-07-01 00:00:00|27861.9   |27861.9   |
|1996-08-01 00:00:00|25485.28  |53347.18  |
|1996-09-01 00:00:00|26381.4   |79728.58  |
|1996-10-01 00:00:00|37515.73  |117244.31 |
|1996-11-01 00:00:00|45600.05  |162844.36 |
|1996-12-01 00:00:00|45239.63  |208083.99 |
|1997-01-01 00:00:00|61258.08  |269342.07 |
|1997-02-01 00:00:00|38483.64  |307825.71 |
|1997-03-01 00:00:00|38547.23  |346372.94 |
|1997-04-01 00:00:00|53032.95  |399405.89 |
|1997-05-01 00:00:00|53781.3   |453187.19 |
|1997-06-01 00:00:00|36362.82  |489550.01 |
|1997-07-01 00:00:00|51020.88  |540570.89 |
|1997-08-01 00:00:00|47287.68  |587858.57 |
|1997-09-01 00:00:00|55629.27  |643487.84 |
|1997-10-01 00:00:00|66749.24  |710237.08 |
|1997-11-01 00:00:00|43533.81  |753770.89 |
|1997-12-01 00:00:00|71398.44  |825169.33 |
|1998-01-01 00:00:00|94222.13  |919391.46 |
|1998-02-01 00:00:00|99415.29  |

## Q31 Tri et limites

In [26]:
df_top_produits_quantite = (
    df_orders_enriched
    .groupBy(
        "product_id",
        "product_name",
    )
    .agg(
        F.sum("quantite").alias(
            "quantite_totale"
        )
    )
    .orderBy(F.desc("quantite_totale"))
    .limit(5)
)

df_top_produits_quantite.show(
    truncate=False
)

+----------+----------------------+---------------+
|product_id|product_name          |quantite_totale|
+----------+----------------------+---------------+
|60        |Camembert Pierrot     |1504           |
|59        |Raclette Courdavault  |1496           |
|31        |Gorgonzola Telino     |1377           |
|56        |Gnocchi di nonna Alice|1263           |
|75        |Rhönbräu Klosterbier  |1151           |
+----------+----------------------+---------------+



In [27]:
df_top_pays = (
    df_orders_enriched
    .groupBy("customer_country")
    .agg(
        F.round(
            F.sum("sous_total"),
            2
        ).alias("ca_total")
    )
    .orderBy(F.desc("ca_total"))
    .limit(3)
)

df_top_pays.show(truncate=False)

+----------------+---------+
|customer_country|ca_total |
+----------------+---------+
|USA             |243618.93|
|GERMANY         |227796.71|
|AUSTRIA         |118104.95|
+----------------+---------+



## Q32 — Écriture du DataFrame enrichi en Parquet

In [28]:
PARQUET_PATH = (
    f"{OUTPUT_PATH}/orders_enriched.parquet"
)

(
    df_orders_enriched
    .write
    .mode("overwrite")
    .parquet(PARQUET_PATH)
)

print(
    "Parquet créé :",
    PARQUET_PATH
)

Parquet créé : /home/jovyan/data/output/orders_enriched.parquet


# Notebook 4- Ecriture Parquet et chargement PosgreSQL bonus

## Q33 — Relecture du fichier Parquet

In [29]:
df_parquet_verification = spark.read.parquet(
    PARQUET_PATH
)

nombre_original = df_orders_enriched.count()
nombre_parquet = df_parquet_verification.count()

print("Lignes du DataFrame original :", nombre_original)
print("Lignes du fichier Parquet :", nombre_parquet)
print("Nombres identiques :", nombre_original == nombre_parquet)

df_parquet_verification.printSchema()

Lignes du DataFrame original : 2082
Lignes du fichier Parquet : 2082
Nombres identiques : True
root
 |-- shipper_id: integer (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- prix_unitaire: double (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- sous_total: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (nullable = true)
 |-- ship_postal_code: string (nullable = true)
 |-- ship_country: string (nullable = true)
 |-- is_shipped: boolean (nullable = true)
 |-- customer_company_name: string (nullable = tru

## Q34- Comparer CSV vs Parquet

In [30]:
CSV_COMPARISON_PATH = (
    f"{OUTPUT_PATH}/orders_enriched_csv"
)

(
    df_orders_enriched
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(CSV_COMPARISON_PATH)
)

In [31]:
def taille_dossier(chemin):
    taille = 0

    for dossier, _, fichiers in os.walk(chemin):
        for fichier in fichiers:
            chemin_fichier = os.path.join(
                dossier,
                fichier
            )

            taille += os.path.getsize(
                chemin_fichier
            )

    return taille

taille_csv = taille_dossier(
    CSV_COMPARISON_PATH
)

taille_parquet = taille_dossier(
    PARQUET_PATH
)

print(
    "Taille CSV :",
    round(taille_csv / 1024, 2),
    "Ko"
)

print(
    "Taille Parquet :",
    round(taille_parquet / 1024, 2),
    "Ko"
)

if taille_csv > 0:
    gain = (
        1 - taille_parquet / taille_csv
    ) * 100

    print(
        "Gain de compression :",
        round(gain, 2),
        "%"
    )

Taille CSV : 1005.35 Ko
Taille Parquet : 1431.22 Ko
Gain de compression : -42.36 %


### Comparaison CSV et Parquet

Parquet est généralement plus compact que CSV grâce à sa compression et à
son stockage en colonnes.

Il préserve également les types de données et permet à Spark de lire
uniquement les colonnes nécessaires. CSV stocke toutes les valeurs sous
forme de texte et nécessite une nouvelle inférence du schéma à chaque
lecture.

## Q35 — Partitionnement par pays

In [32]:
PARTITION_PATH = (
    f"{OUTPUT_PATH}/orders_by_country"
)

df_partition_country = (
    df_orders_enriched
    .withColumn(
        "country",
        F.col("customer_country")
    )
)

(
    df_partition_country
    .write
    .mode("overwrite")
    .partitionBy("country")
    .parquet(PARTITION_PATH)
)

print("Parquet partitionné créé :", PARTITION_PATH)

Parquet partitionné créé : /home/jovyan/data/output/orders_by_country


In [33]:
dossiers_pays = sorted(
    dossier
    for dossier in os.listdir(PARTITION_PATH)
    if dossier.startswith("country=")
)

print("Nombre de partitions :", len(dossiers_pays))

for dossier in dossiers_pays:
    print(dossier)

Nombre de partitions : 21
country=ARGENTINA
country=AUSTRIA
country=BELGIUM
country=BRAZIL
country=CANADA
country=DENMARK
country=FINLAND
country=FRANCE
country=GERMANY
country=IRELAND
country=ITALY
country=MEXICO
country=NORWAY
country=POLAND
country=PORTUGAL
country=SPAIN
country=SWEDEN
country=SWITZERLAND
country=UK
country=USA
country=VENEZUELA


## Q36 Chargement PostgreSQL via JDBC

In [34]:
print(
    "Packages Spark :",
    spark.sparkContext.getConf().get(
        "spark.jars.packages",
        "non défini"
    )
)

Packages Spark : non défini


In [35]:
jdbc_url = (
    "jdbc:postgresql://postgres:5432/tradecorp"
)

proprietes_postgres = {
    "user": "postgres",
    "password": "postgres",
    "driver": "org.postgresql.Driver",
}

(
    df_orders_enriched
    .write
    .mode("overwrite")
    .jdbc(
        url=jdbc_url,
        table="orders_enriched",
        properties=proprietes_postgres,
    )
)

print(
    "Table PostgreSQL orders_enriched créée."
)

Table PostgreSQL orders_enriched créée.


In [36]:
import sys

if "/home/jovyan/src" not in sys.path:
    sys.path.append("/home/jovyan/src")

from reader import read_csv

df_test_reader = read_csv(
    spark,
    "/home/jovyan/data",
    "customers.csv",
)

print("Nombre de clients :", df_test_reader.count())
df_test_reader.show(5, truncate=False)

Nombre de clients : 91
+-----------+----------------------------------+------------------+--------------------+-----------------------------+-----------+------+-----------+-------+--------------+--------------+
|customer_id|company_name                      |contact_name      |contact_title       |address                      |city       |region|postal_code|country|phone         |fax           |
+-----------+----------------------------------+------------------+--------------------+-----------------------------+-----------+------+-----------+-------+--------------+--------------+
|ALFKI      |Alfreds Futterkiste               |Maria Anders      |Sales Representative|Obere Str. 57                |Berlin     |NULL  |12209      |Germany|030-0074321   |030-0076545   |
|ANATR      |Ana Trujillo Emparedados y helados|Ana Trujillo      |Owner               |Avda. de la Constitución 2222|México D.F.|NULL  |05021      |Mexico |(5) 555-4729  |(5) 555-3745  |
|ANTON      |Antonio Moreno Taquería 